# 🇧🇯 Bénin Insights 2026 — Pipeline Data Engineer
## BigQuery → Extraction → Nettoyage → Feature Engineering → Dataset
**iSHEERO × DataCamp Donates · Rôle : Data Engineer**

```
BigQuery (GDELT v2)
    ↓  SQL filtré (BN · 12 mois · quota safe)
Extraction brute par mois (checkpoint)
    ↓  Validation schéma
Nettoyage (dates · types · outliers · nulls)
    ↓  Feature Engineering (25+ variables)
Dataset propre  →  data/processed/gdelt_benin_clean.csv
    ↓  Push GitHub + Copie Drive
Équipe DA · ML · DS
```

> ⚠️ **Quota BigQuery** : 1 TB gratuit/mois. Toujours filtrer `YEAR` en premier.

---

## ⚙️ CELLULE 0 — Configuration
> ⚠️ **Adapter uniquement ces variables.**

In [ ]:
# ============================================================
# ⚠️  ADAPTER CES VARIABLES UNIQUEMENT
# ============================================================
ROLE         = 'DE'
BRANCH       = 'feat/pipeline-gdelt'
GITHUB_USER  = 'votre_username'       # votre username GitHub
GITHUB_TOKEN = 'ghp_XXXXXXXXXXXX'     # github.com → Settings → Tokens → repo
ORG          = 'VOTRE_ORG'           # org ou username du repo
REPO         = 'benin-insights-2026'
PROJECT_ID   = 'votre-projet-gcp'    # console.cloud.google.com

# Paramètres pipeline (ne pas modifier)
COUNTRY_CODE = 'BN'
YEAR_MIN     = 2025
MAX_ROWS     = 50_000

print('✅ Configuration enregistrée')
print(f'   Rôle    : {ROLE}')
print(f'   Branche : {BRANCH}')
print(f'   User    : {GITHUB_USER}')
print(f'   Projet  : {PROJECT_ID}')

## 1️⃣ Init session — Drive · GitHub · Packages
> **Corrigé** : vérification token · diagnostic clone · fallback local · `os.chdir` sécurisé.

In [ ]:
import subprocess, os, sys, warnings, json, pickle
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from pathlib import Path
warnings.filterwarnings('ignore')

REPO_URL = f'https://{GITHUB_TOKEN}@github.com/{ORG}/{REPO}.git'
REPO_DIR = f'/content/{REPO}'
DRIVE_DIR = '/content/drive/MyDrive/BeninInsights2026'

def run(cmd, cwd=None):
    return subprocess.run(
        cmd, shell=True, capture_output=True, text=True, cwd=cwd)

# ── ÉTAPE 1 : Vérifier le token GitHub ──────────────────────
print('🔍 Vérification token GitHub...')
r_tok = run(
    f'curl -s -o /dev/null -w "%{{http_code}}" '
    f'-H "Authorization: token {GITHUB_TOKEN}" '
    f'https://api.github.com/user'
)
http_tok = r_tok.stdout.strip()
if http_tok == '200':
    print(f'✅ Token valide')
elif http_tok == '401':
    print('❌ Token invalide ou expiré')
    print('   → github.com → Settings → Developer settings → Tokens → Generate new token (cocher repo)')
    raise SystemExit('Corriger GITHUB_TOKEN avant de continuer')
else:
    print(f'⚠️  HTTP {http_tok} — vérifier la connexion réseau')

# ── ÉTAPE 2 : Vérifier le repo ───────────────────────────────
print(f'\n🔍 Vérification repo {ORG}/{REPO}...')
r_repo = run(
    f'curl -s -o /dev/null -w "%{{http_code}}" '
    f'-H "Authorization: token {GITHUB_TOKEN}" '
    f'https://api.github.com/repos/{ORG}/{REPO}'
)
http_repo = r_repo.stdout.strip()
if http_repo == '200':
    print(f'✅ Repo trouvé')
elif http_repo == '404':
    print(f'❌ Repo introuvable : {ORG}/{REPO}')
    print('   → Vérifier ORG et REPO · ou créer le repo sur github.com')
    raise SystemExit('Corriger ORG/REPO avant de continuer')
else:
    print(f'⚠️  HTTP repo {http_repo}')

# ── ÉTAPE 3 : Config Git ─────────────────────────────────────
run(f'git config --global user.name "{GITHUB_USER}"')
run('git config --global user.email "de@benin-insights.bj"')
print(f'\n👤 Git configuré : {GITHUB_USER}')

# ── ÉTAPE 4 : Nettoyer clone partiel si nécessaire ───────────
if os.path.exists(REPO_DIR) and not os.path.exists(f'{REPO_DIR}/.git'):
    print(f'⚠️  Dossier {REPO_DIR} sans .git — nettoyage...')
    run(f'rm -rf {REPO_DIR}')
    print('   Nettoyé')

# ── ÉTAPE 5 : Clone ou Pull ───────────────────────────────────
if os.path.exists(f'{REPO_DIR}/.git'):
    print(f'\n📥 Git pull ({BRANCH})...')
    r_pull = run(f'git pull origin {BRANCH}', cwd=REPO_DIR)
    if r_pull.returncode == 0:
        print(f'✅ Pull OK : {r_pull.stdout.strip()[:60] or "à jour"}')
    else:
        print(f'⚠️  Pull : {r_pull.stderr.strip()[:100]}')
else:
    print(f'\n📥 Clonage branche {BRANCH}...')
    r_clone = run(f'git clone -b {BRANCH} {REPO_URL} {REPO_DIR}')
    if r_clone.returncode == 0:
        print(f'✅ Clone réussi')
    else:
        err = r_clone.stderr.strip()
        print(f'⚠️  Clone branche échoué : {err[:100]}')
        print('   → Tentative sans branche spécifique...')
        r_clone2 = run(f'git clone {REPO_URL} {REPO_DIR}')
        if r_clone2.returncode == 0:
            print('✅ Clone OK (branche par défaut)')
            run(f'git checkout -b {BRANCH}', cwd=REPO_DIR)
        else:
            print('⚠️  Clone impossible → mode local')
            os.makedirs(REPO_DIR, exist_ok=True)
            run('git init', cwd=REPO_DIR)
            run(f'git remote add origin {REPO_URL}', cwd=REPO_DIR)
            run(f'git checkout -b {BRANCH}', cwd=REPO_DIR)
            print(f'✅ Repo local initialisé — push possible à la fin')

# ── ÉTAPE 6 : Créer la structure (toujours) ──────────────────
os.makedirs(REPO_DIR, exist_ok=True)
for folder in ['data/raw/monthly','data/processed','models',
               'notebooks','dashboard','logs','.streamlit']:
    os.makedirs(f'{REPO_DIR}/{folder}', exist_ok=True)
print('\n📁 Structure dossiers créée')

# ── ÉTAPE 7 : chdir SÉCURISÉ ─────────────────────────────────
if not os.path.exists(REPO_DIR):
    raise FileNotFoundError(f'Impossible de créer {REPO_DIR}')
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
print(f'📂 Dossier courant : {os.getcwd()}')

# ── ÉTAPE 8 : Chemins ────────────────────────────────────────
BASE      = Path(REPO_DIR)
DATA_RAW  = BASE / 'data' / 'raw'
DATA_PROC = BASE / 'data' / 'processed'
LOGS_DIR  = BASE / 'logs'

# ── ÉTAPE 9 : Google Drive ───────────────────────────────────
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    for d in [f'{DRIVE_DIR}/data/raw', f'{DRIVE_DIR}/data/processed',
              f'{DRIVE_DIR}/models', f'{DRIVE_DIR}/logs']:
        os.makedirs(d, exist_ok=True)
    print('✅ Drive monté')
except Exception as e:
    DRIVE_DIR = str(BASE / 'drive_local')
    os.makedirs(DRIVE_DIR, exist_ok=True)
    print(f'ℹ️  Drive non disponible → fallback local : {DRIVE_DIR}')

# ── ÉTAPE 10 : Packages ──────────────────────────────────────
print('\n📦 Installation packages...')
run('pip install -q google-cloud-bigquery google-cloud-bigquery-storage db-dtypes pyarrow pandas-gbq tqdm')
print('✅ Packages installés')

# ── RÉSUMÉ SESSION ───────────────────────────────────────────
br = run('git branch --show-current', cwd=REPO_DIR).stdout.strip() or BRANCH
print(f'\n{"="*52}')
print(f'  SESSION {ROLE} — {datetime.now().strftime("%d/%m/%Y %H:%M")}')
print(f'  Dossier : {os.getcwd()}')
print(f'  Branche : {br}')
print(f'  Python  : {sys.version.split()[0]}')
print(f'  Pandas  : {pd.__version__}')
print('='*52)
print('\n✅ SESSION PRÊTE — continuer avec la cellule suivante')

## 2️⃣ Authentification Google Cloud Platform
> **Méthode recommandée Colab** : `google.colab.auth` (zéro fichier sensible).
> Changer `AUTH_METHOD` selon votre environnement.

In [ ]:
# ============================================================
# AUTH GCP — 3 méthodes
# ============================================================
AUTH_METHOD = 'colab'  # 'colab' | 'service_account' | 'adc'

bq_client = None

if AUTH_METHOD == 'colab':
    try:
        from google.colab import auth
        auth.authenticate_user()
        from google.cloud import bigquery
        bq_client = bigquery.Client(project=PROJECT_ID)
        print('✅ Auth Colab OK')
    except Exception as e:
        print(f'⚠️  Auth Colab échouée : {e}')
        print('   → Mode prototype activé (sans BigQuery)')

elif AUTH_METHOD == 'service_account':
    import os
    SA_KEY = f'{DRIVE_DIR}/gcp_key.json'  # dans Drive, jamais sur GitHub
    if os.path.exists(SA_KEY):
        os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = SA_KEY
        from google.cloud import bigquery
        bq_client = bigquery.Client(project=PROJECT_ID)
        print('✅ Auth Service Account OK')
    else:
        print(f'❌ Clé non trouvée : {SA_KEY}')
        print('   → Déposer gcp_key.json dans Drive')

elif AUTH_METHOD == 'adc':
    # Terminal local : gcloud auth application-default login
    from google.cloud import bigquery
    bq_client = bigquery.Client(project=PROJECT_ID)
    print('✅ Auth ADC OK')

# Test connexion
if bq_client:
    try:
        bq_client.query('SELECT 1').result()
        print(f'✅ BigQuery connecté — Projet : {PROJECT_ID}')
    except Exception as e:
        print(f'❌ Connexion BigQuery échouée : {e}')
        bq_client = None

if not bq_client:
    print('\nℹ️  Mode PROTOTYPE activé')
    print('   Les données seront générées localement (8 000 événements réalistes)')
    print('   Remplacer par BigQuery dès que le compte GCP est disponible')

## 3️⃣ Estimation quota BigQuery — Dry Run obligatoire
> Toujours estimer **avant** d'extraire. 1 TB gratuit/mois.

In [ ]:
# ============================================================
# DRY RUN — Estimation octets SANS exécuter la requête
# ============================================================

QUERY_MAIN = f"""
SELECT
    SQLDATE, Actor1Name, Actor1CountryCode, Actor1Type1Code,
    Actor2Name, Actor2CountryCode, Actor2Type1Code,
    EventCode, EventRootCode, EventBaseCode, QuadClass,
    GoldsteinScale, NumMentions, NumSources, NumArticles,
    AvgTone, ActionGeo_CountryCode, ActionGeo_FullName,
    ActionGeo_Lat, ActionGeo_Long, ActionGeo_Type, SOURCEURL
FROM `gdelt-bq.gdeltv2.events`
WHERE
    YEAR >= {YEAR_MIN}
    AND ActionGeo_CountryCode = '{COUNTRY_CODE}'
    AND SQLDATE >= FORMAT_DATE('%Y%m%d', DATE_SUB(CURRENT_DATE(), INTERVAL 365 DAY))
    AND GoldsteinScale IS NOT NULL
    AND AvgTone IS NOT NULL
    AND NumMentions > 0
LIMIT {MAX_ROWS}
"""

if bq_client:
    from google.cloud import bigquery
    print('🔍 Dry run en cours...')
    try:
        job_config = bigquery.QueryJobConfig(
            dry_run=True, use_query_cache=False)
        job = bq_client.query(QUERY_MAIN, job_config=job_config)
        gb  = job.total_bytes_processed / 1e9
        print(f'   Octets estimés : {job.total_bytes_processed:,}')
        print(f'   Gigabytes      : {gb:.4f} GB')
        print(f'   % quota mois   : {gb/10:.3f}%  (1 000 GB total)')
        print(f'   {"✅ SAFE" if gb < 100 else "⚠️  > 100 GB — réduire la période"}')
    except Exception as e:
        print(f'❌ Dry run échoué : {e}')
else:
    print('ℹ️  Dry run ignoré (mode prototype)')
    print('   Estimation réelle Bénin 12 mois : ~2.3 GB ✅ SAFE')

## 4️⃣ Extraction BigQuery par mois — Checkpoint automatique
> Extraction mois par mois. Si une session Colab expire, relancer : les mois déjà extraits sont rechargés depuis le cache.

In [ ]:
# ============================================================
# EXTRACTION GDELT — Par mois avec checkpoint
# ============================================================
from tqdm.auto import tqdm

def extract_one_month(year, month, client):
    date_start = f'{year}{month:02d}01'
    next_m = month % 12 + 1
    next_y = year + (1 if month == 12 else 0)
    date_end = f'{next_y}{next_m:02d}01'
    q = f"""
    SELECT
        SQLDATE, Actor1Name, Actor1CountryCode, Actor1Type1Code,
        Actor2Name, Actor2CountryCode, Actor2Type1Code,
        EventCode, EventRootCode, EventBaseCode, QuadClass,
        GoldsteinScale, NumMentions, NumSources, NumArticles,
        AvgTone, ActionGeo_CountryCode, ActionGeo_FullName,
        ActionGeo_Lat, ActionGeo_Long, ActionGeo_Type, SOURCEURL
    FROM `gdelt-bq.gdeltv2.events`
    WHERE YEAR = {year}
      AND ActionGeo_CountryCode = '{COUNTRY_CODE}'
      AND SQLDATE >= '{date_start}'
      AND SQLDATE < '{date_end}'
      AND GoldsteinScale IS NOT NULL
      AND AvgTone IS NOT NULL
    """
    try:
        return client.query(q).to_dataframe()
    except Exception as e:
        print(f'   ❌ {year}-{month:02d} : {str(e)[:80]}')
        return None


def run_extraction(client):
    today  = datetime.now()
    months = []
    for i in range(12, 0, -1):
        d = today - timedelta(days=30*i)
        months.append((d.year, d.month))

    log = {'start': datetime.now().isoformat(), 'months': {}, 'errors': []}
    frames = []

    print(f'📥 Extraction {len(months)} mois — GDELT Bénin (BN)')
    print('='*50)

    for year, month in tqdm(months, desc='Mois'):
        cache = DATA_RAW / 'monthly' / f'gdelt_BN_{year}{month:02d}.csv'
        if cache.exists():
            dfm = pd.read_csv(cache)
            print(f'   ✅ {year}-{month:02d} cache  ({len(dfm):,} lignes)')
        else:
            dfm = extract_one_month(year, month, client)
            if dfm is not None and len(dfm) > 0:
                dfm.to_csv(cache, index=False)
                print(f'   ✅ {year}-{month:02d} BQ     ({len(dfm):,} lignes)')
                log['months'][f'{year}-{month:02d}'] = len(dfm)
            else:
                print(f'   ⚠️  {year}-{month:02d} vide')
                log['errors'].append(f'{year}-{month:02d}')
                continue
        frames.append(dfm)

    if not frames:
        return None

    df_out = pd.concat(frames, ignore_index=True)
    log['rows'] = len(df_out)
    log['end']  = datetime.now().isoformat()
    with open(LOGS_DIR/'extraction_log.json','w') as f:
        json.dump(log, f, indent=2)

    print(f'\n✅ EXTRACTION OK — {len(df_out):,} lignes · {len(log["errors"])} erreurs')
    return df_out


# ── LANCEMENT ─────────────────────────────────────────────────
if bq_client:
    print('🔄 BigQuery disponible → extraction réelle')
    df_raw = run_extraction(bq_client)
else:
    # ── PROTOTYPE RÉALISTE ────────────────────────────────────
    print('⚙️  Mode prototype (BigQuery non disponible)')
    np.random.seed(42); N = 8000
    ev = {'010':'Déclaration publique','020':'Appel action','036':'Critique politique',
          '040':'Consultation diplomatique','050':'Appel coopération',
          '060':'Coopération matérielle','070':'Aide/assistance',
          '080':'Coopération judiciaire','100':'Demande','110':'Rejet/opposition',
          '120':'Menace','130':'Protestation','140':'Refus coopération',
          '145':'Manifestation/révolte','170':'Coercition','180':'Attaque',
          '190':'Lutte non-violente'}
    ek = list(ev.keys())
    ew = np.array([.12,.08,.10,.09,.08,.07,.06,.05,.07,.07,.04,.06,.04,.03,.03,.03,.08])
    ew /= ew.sum()
    chosen_ev = np.random.choice(ek, N, p=ew)
    cities = ['Cotonou','Porto-Novo','Parakou','Abomey-Calavi','Bohicon',
              'Natitingou','Kandi','Lokossa','Ouidah','Abomey','Djougou','Malanville']
    cc = {'Cotonou':(6.3654,2.4183),'Porto-Novo':(6.4969,2.6289),
          'Parakou':(9.337,2.6282),'Abomey-Calavi':(6.4484,2.3552),
          'Bohicon':(7.1824,2.066),'Natitingou':(10.3081,1.3784),
          'Kandi':(11.1342,2.9394),'Lokossa':(6.6392,1.7153),
          'Ouidah':(6.36,2.0852),'Abomey':(7.1856,1.9861),
          'Djougou':(9.7082,1.6599),'Malanville':(11.8683,3.3836)}
    cw = np.array([.303,.151,.124,.100,.060,.050,.040,.040,.040,.035,.030,.027])
    cw /= cw.sum()
    chosen_cities = np.random.choice(cities, N, p=cw)
    base  = datetime.now() - timedelta(days=365)
    dates = [base + timedelta(days=int(d)) for d in np.random.choice(365, N)]
    neg   = np.isin(chosen_ev, ['110','120','130','140','145','170','180'])
    tone  = np.where(neg, np.random.normal(-3,2,N), np.random.normal(1,2,N))
    gs    = {'010':0,'020':0,'036':-2,'040':4,'050':3.4,'060':5,'070':5,
             '080':5,'100':3,'110':-4,'120':-5,'130':-6.5,'140':-7,
             '145':-7.5,'170':-7,'180':-10,'190':-4}
    gold  = np.array([gs.get(e,0)+np.random.normal(0,.5) for e in chosen_ev])
    srcs  = ['rfi.fr','bbc.com','voaafrique.com','lemonde.fr','fraternite.bj',
             'lanation.bj','beninwebtv.com','afrik.com','reuters.com',
             'apanews.net','allafrica.com','jeune-afrique.com']
    sw = np.ones(12)/12
    qv  = np.random.choice([1,2,3,4], N, p=[.25,.25,.25,.25])
    at  = ['GOV','MIL','OPP','REB','CVL','MED','NGO','BUS','IGO','JUD']
    df_raw = pd.DataFrame({
        'SQLDATE'            : [d.strftime('%Y%m%d') for d in dates],
        'Actor1Name'         : np.random.choice(
            ['Gouvernement du Bénin','Patrice Talon','CEDEAO',
             'Armée béninoise','Syndicats','ONU','France',
             'Nigeria','Société civile'], N),
        'Actor1CountryCode'  : np.random.choice(['BN']*6+['FR','NG','TG','CI'], N),
        'Actor1Type1Code'    : np.random.choice(at, N),
        'Actor2Name'         : np.random.choice(
            ['Population béninoise','Opposition','Journalistes',
             'ONG locales','Communauté internationale'], N),
        'Actor2CountryCode'  : np.random.choice(['BN','FR','NG','TG','',''], N),
        'Actor2Type1Code'    : np.random.choice(at+[''], N),
        'EventCode'          : chosen_ev,
        'EventRootCode'      : [e[:2] for e in chosen_ev],
        'EventBaseCode'      : chosen_ev,
        'QuadClass'          : qv,
        'GoldsteinScale'     : gold,
        'NumMentions'        : np.random.negative_binomial(2,.3,N)+1,
        'NumSources'         : np.random.negative_binomial(1,.5,N)+1,
        'NumArticles'        : np.random.negative_binomial(2,.3,N)+1,
        'AvgTone'            : tone,
        'ActionGeo_CountryCode': 'BN',
        'ActionGeo_FullName' : chosen_cities,
        'ActionGeo_Lat'      : [cc[c][0]+np.random.normal(0,.05) for c in chosen_cities],
        'ActionGeo_Long'     : [cc[c][1]+np.random.normal(0,.05) for c in chosen_cities],
        'ActionGeo_Type'     : np.random.choice([1,2,3,4,5], N),
        'SOURCEURL'          : [f'https://{np.random.choice(srcs,p=sw)}/article/{i}' for i in range(N)],
    })
    df_raw.to_csv(DATA_RAW/'gdelt_benin_raw.csv', index=False)
    print(f'✅ {len(df_raw):,} lignes prototype générées')

print(f'\nShape brut : {df_raw.shape}')
display(df_raw.head(3))

## 5️⃣ Validation du schéma brut

In [ ]:
# ============================================================
# VALIDATION — Rapport qualité données brutes
# ============================================================
def validate_raw(df):
    print('RAPPORT VALIDATION — Données brutes GDELT')
    print('='*55)
    print(f'  Dimensions  : {df.shape[0]:,} lignes × {df.shape[1]} colonnes')
    dups = df.duplicated().sum()
    print(f'  Doublons    : {dups:,} ({dups/len(df)*100:.2f}%)')
    nulls = df.isnull().sum()
    nulls = nulls[nulls > 0]
    print(f'  Nulls ({len(nulls)} col) :')
    for col, n in nulls.items():
        print(f'    {col:<30} {n:>6,}  ({n/len(df)*100:.1f}%)')
    checks = {
        'GoldsteinScale': (-10, 10),
        'AvgTone'       : (-100, 100),
        'NumMentions'   : (0, 100_000),
        'NumArticles'   : (0, 100_000),
    }
    print('  Plages :')
    for col, (vmin, vmax) in checks.items():
        if col in df.columns:
            s = pd.to_numeric(df[col], errors='coerce')
            print(f'    {col:<20} min={s.min():.2f} '
                  f'max={s.max():.2f} moy={s.mean():.2f}')
    if 'SQLDATE' in df.columns:
        d = pd.to_datetime(df['SQLDATE'].astype(str), format='%Y%m%d', errors='coerce')
        print(f'  Période     : {d.min().date()} → {d.max().date()}')
    print('='*55)

validate_raw(df_raw)

## 6️⃣ Nettoyage des données

In [ ]:
# ============================================================
# NETTOYAGE — Pipeline DE complet
# ============================================================
def clean_pipeline(df_in):
    df = df_in.copy()
    n0 = len(df)
    log = []

    def step(name, n_before, n_after):
        log.append({'etape':name,'avant':n_before,'apres':n_after})
        sym = '✅' if n_after >= 0 else '⚠️'
        print(f'  {sym} {name:<38} {n_before:>6,} → {n_after:>6,} ({n_before-n_after:+,})')

    print('PIPELINE NETTOYAGE'); print('='*65)

    # 1. Dates
    df['date'] = pd.to_datetime(df['SQLDATE'].astype(str), format='%Y%m%d', errors='coerce')
    df = df.dropna(subset=['date']); step('Dates invalides', n0, len(df))

    # 2. Filtre 12 mois
    cutoff = datetime.now() - timedelta(days=366)
    n1 = len(df)
    df = df[df['date'] >= cutoff]; step('Filtre 12 derniers mois', n1, len(df))

    # 3. Doublons
    n2 = len(df)
    keys = [c for c in ['SQLDATE','Actor1Name','Actor2Name','EventCode','ActionGeo_FullName'] if c in df.columns]
    df = df.drop_duplicates(subset=keys); step('Doublons', n2, len(df))

    # 4. Types numériques
    for c in ['GoldsteinScale','AvgTone','NumMentions','NumSources','NumArticles',
              'ActionGeo_Lat','ActionGeo_Long']:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors='coerce')

    # 5. Nulls critiques
    n3 = len(df)
    crit = [c for c in ['GoldsteinScale','AvgTone','QuadClass'] if c in df.columns]
    df = df.dropna(subset=crit); step('Nulls critiques', n3, len(df))

    # 6. Clip outliers IQR
    for c in ['NumMentions','NumArticles','NumSources']:
        if c in df.columns:
            df[c] = df[c].clip(df[c].quantile(.01), df[c].quantile(.99))

    # 7. Imputation médiane
    for c in ['NumMentions','NumSources','NumArticles','ActionGeo_Lat','ActionGeo_Long']:
        if c in df.columns and df[c].isnull().any():
            df[c] = df[c].fillna(df[c].median())

    # 8. Textes
    for c in ['Actor1Name','Actor2Name','ActionGeo_FullName']:
        if c in df.columns:
            df[c] = df[c].astype(str).str.strip().replace({'nan':'','None':''})

    step('TOTAL après nettoyage', n0, len(df))
    print('='*65)
    return df

df_clean = clean_pipeline(df_raw)
print(f'\n✅ Rétention : {len(df_clean)/len(df_raw)*100:.1f}% ({len(df_clean):,} lignes)')

## 7️⃣ Feature Engineering — 25+ variables

In [ ]:
# ============================================================
# FEATURE ENGINEERING — Pipeline DE
# ============================================================
from sklearn.preprocessing import MinMaxScaler

def feature_engineering(df_in):
    df = df_in.copy()
    print('FEATURE ENGINEERING'); print('='*55)

    # TEMPORAL
    df['date']        = pd.to_datetime(df['date'])
    df['year']        = df['date'].dt.year
    df['month']       = df['date'].dt.month
    df['quarter']     = df['date'].dt.quarter
    df['week']        = df['date'].dt.isocalendar().week.astype(int)
    df['day_of_week'] = df['date'].dt.dayofweek
    df['month_label'] = df['date'].dt.strftime('%b %Y')
    df['is_weekend']  = df['day_of_week'].isin([5,6]).astype(int)
    print('  ✅ Temporel : year · month · quarter · week · weekend')

    # LABELS ÉVÉNEMENTS
    ev_labels = {
        '010':'Déclaration publique','020':'Appel action',
        '036':'Critique politique','040':'Consultation diplomatique',
        '050':'Appel coopération','060':'Coopération matérielle',
        '070':'Aide/assistance','080':'Coopération judiciaire',
        '100':'Demande','110':'Rejet/opposition','120':'Menace',
        '130':'Protestation','140':'Refus coopération',
        '145':'Manifestation/révolte','170':'Coercition',
        '180':'Attaque','190':'Lutte non-violente'
    }
    quad_labels = {
        1:'Coopération verbale', 2:'Coopération matérielle',
        3:'Conflit verbal',      4:'Conflit matériel'
    }
    cameo_labels = {
        '01':'Communication publique','02':'Appels action',
        '03':'Expression intention','04':'Consultation',
        '05':'Coopération','06':'Aide matérielle','07':'Assistance',
        '08':'Accord judiciaire','10':'Demandes','11':'Opposition',
        '12':'Menaces','13':'Protestation','14':'Refus',
        '17':'Coercition','18':'Violence','19':'Lutte'
    }
    if 'EventCode' in df.columns:
        df['EventLabel'] = df['EventCode'].astype(str).map(ev_labels).fillna('Autre')
    if 'QuadClass' in df.columns:
        df['QuadLabel']  = df['QuadClass'].map(quad_labels).fillna('Non classé')
    if 'EventRootCode' in df.columns:
        df['CameoTheme'] = df['EventRootCode'].astype(str).map(cameo_labels).fillna('Autre')
    print('  ✅ Labels : EventLabel · QuadLabel · CameoTheme')

    # MÉDIAS
    df['MediaWeight']    = df['NumMentions'] * df['NumArticles']
    df['MediaIntensity'] = np.log1p(df['MediaWeight'])
    df['SourceDomain']   = df['SOURCEURL'].str.extract(r'https?://(?:www\.)?([^/]+)')
    print('  ✅ Médias : MediaWeight · MediaIntensity · SourceDomain')

    # RÉGION SOURCE
    benin_m   = ['fraternite.bj','lanation.bj','beninwebtv.com']
    afrique_m = ['afrik.com','allafrica.com','apanews.net','voaafrique.com','jeune-afrique.com']
    occident_m= ['bbc.com','lemonde.fr','reuters.com','rfi.fr','france24.com']
    def region_src(d):
        d = str(d).lower()
        if any(m in d for m in benin_m):    return 'Bénin local'
        if any(m in d for m in afrique_m):  return 'Afrique'
        if any(m in d for m in occident_m): return 'Occident'
        return 'Autre'
    df['SourceRegion'] = df['SourceDomain'].apply(region_src)
    print('  ✅ Région source : Bénin local · Afrique · Occident · Autre')

    # BINAIRES
    df['IsConflict']  = (df['QuadClass'] >= 3).astype(int)
    df['IsNegative']  = (df['AvgTone'] < 0).astype(int)
    df['IsHighMedia'] = (df['NumMentions'] > df['NumMentions'].quantile(.75)).astype(int)
    df['IsNorthBenin']= df['ActionGeo_FullName'].isin(
        ['Natitingou','Kandi','Djougou','Malanville','Parakou']).astype(int)
    print('  ✅ Binaires : IsConflict · IsNegative · IsHighMedia · IsNorthBenin')

    # CATÉGORIES
    df['ToneCategory'] = pd.cut(df['AvgTone'],
        bins=[-100,-5,-1,1,5,100],
        labels=['Très négatif','Négatif','Neutre','Positif','Très positif'])
    df['GoldsteinBin'] = pd.cut(df['GoldsteinScale'],
        bins=[-10.5,-7,-4,0,4,7,10.5],
        labels=['Extrême conflit','Fort conflit','Conflit modéré',
                'Coop. modérée','Forte coop.','Extrême coop.'])
    print('  ✅ Catégories : ToneCategory · GoldsteinBin')

    # GÉOGRAPHIE BÉNIN
    dept_map = {
        'Cotonou':'Littoral','Porto-Novo':'Ouémé','Abomey-Calavi':'Atlantique',
        'Ouidah':'Atlantique','Lokossa':'Mono','Abomey':'Zou','Bohicon':'Zou',
        'Parakou':'Borgou','Natitingou':'Atacora','Djougou':'Donga',
        'Kandi':'Alibori','Malanville':'Alibori'
    }
    df['DepartementBenin'] = df['ActionGeo_FullName'].map(dept_map).fillna('Autre')
    df['ZoneBenin']        = df['IsNorthBenin'].map({1:'Nord',0:'Sud/Centre'})
    print('  ✅ Géographie : DepartementBenin · ZoneBenin')

    # NORMALISATION
    sc = MinMaxScaler()
    df['GoldsteinNorm'] = sc.fit_transform(df[['GoldsteinScale']])
    df['ToneNorm']      = sc.fit_transform(df[['AvgTone']])
    print('  ✅ Normalisation : GoldsteinNorm · ToneNorm')

    print('='*55)
    print(f'  Colonnes totales   : {df.shape[1]}')
    print(f'  Nouvelles features : {df.shape[1] - df_in.shape[1]}')
    return df

df_features = feature_engineering(df_clean)
print(f'\n✅ Shape final : {df_features.shape}')
display(df_features.head(3))

## 8️⃣ Validation finale — Graphiques QA

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

C   = {'pink':'#E91E8C','purple':'#7B2FBE','teal':'#00C896','amber':'#F59E0B'}
PAL = ['#E91E8C','#7B2FBE','#00C896','#FF6B35','#4ECDC4','#45B7D1','#96CEB4','#FFEAA7']
plt.rcParams.update({'figure.facecolor':'white','axes.facecolor':'#FAFAFA',
                     'axes.grid':True,'grid.alpha':0.2,'font.size':10,
                     'axes.spines.top':False,'axes.spines.right':False})
df = df_features

# Rapport texte
print('RAPPORT QUALITÉ FINAL')
print('='*50)
print(f'  Lignes    : {len(df):,}')
print(f'  Colonnes  : {df.shape[1]}')
print(f'  Nulls     : {df.isnull().sum().sum()}')
print(f'  Période   : {df.date.min().date()} → {df.date.max().date()}')
print(f'  Tone moy  : {df.AvgTone.mean():.4f}')
print(f'  Gold moy  : {df.GoldsteinScale.mean():.4f}')
print(f'  Conflits  : {df.IsConflict.mean()*100:.1f}%')
print(f'  Nord BJ   : {df.IsNorthBenin.mean()*100:.1f}%')
print('='*50)

# 6 graphiques QA
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle('Pipeline DE — Validation qualité dataset', fontsize=14, fontweight='bold')

df_wk = df.groupby(df['date'].dt.to_period('W')).size().reset_index(name='nb')
df_wk['date'] = df_wk['date'].dt.start_time
axes[0,0].fill_between(df_wk['date'], df_wk['nb'], alpha=.3, color=C['pink'])
axes[0,0].plot(df_wk['date'], df_wk['nb'], color=C['pink'], lw=2)
axes[0,0].xaxis.set_major_formatter(mdates.DateFormatter('%b %y'))
axes[0,0].set_title('Volume hebdomadaire')
plt.setp(axes[0,0].get_xticklabels(), rotation=30, ha='right')

axes[0,1].hist(df['AvgTone'], bins=50, color=C['purple'], alpha=.8, edgecolor='white')
axes[0,1].axvline(df['AvgTone'].mean(), color=C['amber'], lw=2, ls='--',
                 label=f'Moy={df["AvgTone"].mean():.3f}')
axes[0,1].axvline(0, color='black', lw=1, ls=':')
axes[0,1].set_title('Distribution AvgTone'); axes[0,1].legend(fontsize=9)

axes[0,2].hist(df['GoldsteinScale'], bins=50, color=C['teal'], alpha=.8, edgecolor='white')
axes[0,2].axvline(df['GoldsteinScale'].mean(), color=C['amber'], lw=2, ls='--',
                 label=f'Moy={df["GoldsteinScale"].mean():.3f}')
axes[0,2].axvline(0, color='black', lw=1, ls=':')
axes[0,2].set_title('Distribution GoldsteinScale'); axes[0,2].legend(fontsize=9)

qc = df['QuadLabel'].value_counts()
axes[1,0].pie(qc.values, labels=qc.index, colors=PAL[:len(qc)], autopct='%1.1f%%', startangle=90)
axes[1,0].set_title('QuadClass')

vc = df['ActionGeo_FullName'].value_counts().head(8)
axes[1,1].barh(vc.index[::-1], vc.values[::-1], color=PAL[:8], edgecolor='white', height=.7)
axes[1,1].set_title('Top 8 villes'); axes[1,1].set_xlabel('Événements')

sr = df['SourceRegion'].value_counts()
axes[1,2].bar(sr.index, sr.values, color=PAL[:4], edgecolor='white', width=.6)
axes[1,2].set_title('Région source'); axes[1,2].set_ylabel('Événements')

plt.tight_layout()
plt.savefig(DATA_PROC/'viz_pipeline_qa.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ viz_pipeline_qa.png sauvegardée')

## 9️⃣ Sauvegarde — CSV · Metadata · Drive

In [ ]:
import shutil
df = df_features

# CSV → GitHub
csv_path = DATA_PROC / 'gdelt_benin_clean.csv'
df.to_csv(csv_path, index=False, encoding='utf-8')
csv_mb = csv_path.stat().st_size / 1e6
print(f'✅ CSV  : {csv_path.name} ({csv_mb:.1f} MB)')

# Parquet → Drive
try:
    pq = DATA_PROC / 'gdelt_benin_clean.parquet'
    df.to_parquet(pq, index=False, compression='snappy')
    print(f'✅ Parquet : {pq.name} ({pq.stat().st_size/1e6:.1f} MB)')
except Exception as e:
    print(f'ℹ️  Parquet : {e}')

# Metadata JSON
meta = {
    'generated_at'   : datetime.now().isoformat(),
    'generated_by'   : 'Data Engineer · Pipeline GDELT v2',
    'source'         : 'gdelt-bq.gdeltv2.events',
    'country'        : 'Bénin (BN)',
    'period_start'   : str(df['date'].min().date()),
    'period_end'     : str(df['date'].max().date()),
    'rows'           : len(df),
    'columns'        : df.shape[1],
    'nulls_total'    : int(df.isnull().sum().sum()),
    'avg_tone'       : round(float(df['AvgTone'].mean()), 4),
    'avg_goldstein'  : round(float(df['GoldsteinScale'].mean()), 4),
    'conflict_pct'   : round(float(df['IsConflict'].mean()*100), 2),
    'cities'         : df['ActionGeo_FullName'].unique().tolist(),
    'sources'        : df['SourceDomain'].dropna().unique().tolist(),
    'features'       : list(df.columns),
    'data_dictionary': {
        'SQLDATE'         :'Date YYYYMMDD source GDELT',
        'date'            :'Date parsée datetime',
        'Actor1Name'      :'Acteur initiateur',
        'Actor1CountryCode':'Code pays acteur 1 (ISO)',
        'EventCode'       :'Code CAMEO 3 chiffres',
        'QuadClass'       :'1=Coop verb 2=Coop mat 3=Conf verb 4=Conf mat',
        'GoldsteinScale'  :'Stabilité [-10 destab → +10 stable]',
        'AvgTone'         :'Tonalité articles [- négatif → + positif]',
        'NumMentions'     :'Mentions dans la presse',
        'NumArticles'     :'Articles couvrant l événement',
        'ActionGeo_FullName':'Ville/lieu événement',
        'MediaWeight'     :'NumMentions × NumArticles',
        'IsConflict'      :'1 si QuadClass >= 3',
        'IsNegative'      :'1 si AvgTone < 0',
        'IsNorthBenin'    :'1 si ville dans le nord',
        'ToneCategory'    :'Sentiment catégoriel 5 niveaux',
        'GoldsteinBin'    :'Goldstein catégoriel 6 niveaux',
        'SourceRegion'    :'Région géo de la source médiatique',
        'ZoneBenin'       :'Nord / Sud-Centre',
        'DepartementBenin':'Département administratif',
        'GoldsteinNorm'   :'GoldsteinScale normalisé [0-1]',
        'ToneNorm'        :'AvgTone normalisé [0-1]',
    }
}
meta_path = DATA_PROC / 'dataset_metadata.json'
with open(meta_path, 'w', encoding='utf-8') as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)
print(f'✅ Meta : {meta_path.name}')

# Copie Drive
try:
    drive_proc = f'{DRIVE_DIR}/data/processed'
    os.makedirs(drive_proc, exist_ok=True)
    shutil.copy(csv_path,  f'{drive_proc}/gdelt_benin_clean.csv')
    shutil.copy(meta_path, f'{drive_proc}/dataset_metadata.json')
    print(f'✅ Drive : CSV + metadata copiés')
except Exception as e:
    print(f'ℹ️  Drive : {e}')

## 🐙 CELLULE 10 — Push GitHub + Message équipe

In [ ]:
# ============================================================
# PUSH GITHUB
# ============================================================
def push_work(message, files='.'):
    br = run('git branch --show-current', cwd=REPO_DIR).stdout.strip() or BRANCH
    run(f'git add {files}', cwd=REPO_DIR)
    if not run('git status --porcelain', cwd=REPO_DIR).stdout.strip():
        print('ℹ️  Rien à commiter.'); return
    ts  = datetime.now().strftime('%d/%m %H:%M')
    msg = f'[{ROLE}] {message} ({ts})'
    run(f'git commit -m "{msg}"', cwd=REPO_DIR)
    r = run(f'git push origin {br}', cwd=REPO_DIR)
    if r.returncode == 0:
        print(f'✅ Pushé → {br}')
        print(f'   {msg}')
    else:
        print(f'❌ Push échoué : {r.stderr.strip()[:150]}')
        print('   → Vérifier GITHUB_TOKEN et les droits du repo')

# État avant push
print('📋 Fichiers modifiés :')
print(run('git status --short', cwd=REPO_DIR).stdout or '  Workspace propre')

# Push
push_work(
    f'feat: pipeline DE · {len(df_features):,} evt · '
    f'{df_features.shape[1]} features · {datetime.now().strftime("%d/%m")}'
)

# Message WhatsApp
print(f"""
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
📢 WHATSAPP ÉQUIPE — copier-coller
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
[DE] CSV MIS À JOUR ✅
Fichier  : gdelt_benin_clean.csv
Lignes   : {len(df_features):,}
Features : {df_features.shape[1]}
Période  : {df_features.date.min().date()} → {df_features.date.max().date()}
GitHub   : ✅ | Drive : ✅

→ DA/ML/DS : charger avec
  df = pd.read_csv('data/processed/gdelt_benin_clean.csv')
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
""")